# Extract and analyze faces from pictures

#Setup


In [ ]:
#@title ▶ Install the required tools and setup the environment

!pip install -q deepface

In [ ]:
#@title ▶ Load a dataset (from [Huggingface](https://huggingface.co/datasets))  * Skip if you are using your own images

from datasets import load_dataset

dataset = load_dataset("yuvalkirstain/pexel_people", split='train', streaming=True)

images = iter(dataset.take(100)["image"])

## If you want to use your own images

Check 'use_own_dataset' and run, it will connect to drive, then in the next cell input the folder path where you have the images and run it.

The best way to get the path is to find the folder in the file explorer on the left, hover over it, click on the three dots at the right and select "copy path". Then paste it in the text box.

In [ ]:
#@markdown Select if you want to use your own images
use_own_dataset = False # @param {type:"boolean"}

if use_own_dataset:
  from google.colab import drive
  drive.mount('/content/drive')

In [ ]:
#@markdown Specify the path to your images on Drive

images_path = '/content/drive/MyDrive/tmp/BAU' # @param {type:"string"}

if use_own_dataset:
  from pathlib import Path

  image_extensions = {'.jpg', '.jpeg', '.png', '.webp'}
  image_folder = Path(images_path)

  images = [img for img in image_folder.glob('*')
            if img.suffix.lower() in image_extensions]

  print(f"Found {len(images)} images in {images_path}")

#Processing

In [ ]:
#@title Extract all the faces from the images

from deepface import DeepFace
import cv2
import os
from tqdm.notebook import tqdm
import pathlib

def extract_and_save_faces(image_paths, output_folder="extracted_faces",
                          detector_backend="opencv", align=True):
    """
    Extract all faces from a list of image paths and save them as cropped images.

    Args:
        image_paths (list): List of paths to images
        output_folder (str): Folder where cropped face images will be saved
        detector_backend (str): Face detector to use. Options: 'opencv', 'retinaface',
            'mtcnn', 'ssd', 'dlib', 'mediapipe', 'yolov8', 'yunet', 'centerface'
        align (bool): Whether to align faces
    """
    # Create output folder if it doesn't exist
    os.makedirs(output_folder, exist_ok=True)

    face_count = 0

    for img_idx, img_path in tqdm(enumerate(image_paths)):
        try:
            if type(img_path) != pathlib.PosixPath:
              new_img_path = f"/tmp/dataset_image_{img_idx}.png"
              img_path.save(new_img_path, format="PNG")
              img_path = new_img_path
            else:
              img_path = str(img_path)

            # Extract faces from the image
            face_objs = DeepFace.extract_faces(
                img_path=img_path,
                detector_backend=detector_backend,
                enforce_detection=False,  # Don't raise error if no face found
                align=align,
                grayscale=False
            )

            # Save each detected face
            for face_idx, face_obj in enumerate(face_objs):
                if face_obj["confidence"] < 0.8:
                  continue

                # Get the face image (numpy array)
                face_img = face_obj['face']

                # Convert from RGB to BGR for OpenCV
                face_img_bgr = cv2.cvtColor((face_img * 255).astype('uint8'),
                                           cv2.COLOR_RGB2BGR)

                # Create filename
                img_name = os.path.splitext(os.path.basename(img_path))[0]
                output_filename = f"{img_name}_face_{face_idx}.jpg"
                output_path = os.path.join(output_folder, output_filename)

                # Save the cropped face
                cv2.imwrite(output_path, face_img_bgr)

                face_count += 1
                # print(f"Saved face {face_idx} from {img_path} to {output_path}")

                # Optional: Print facial area coordinates
                # facial_area = face_obj['facial_area']
                # print(f"  Facial area: x={facial_area['x']}, y={facial_area['y']}, "
                #      f"w={facial_area['w']}, h={facial_area['h']}")

        except Exception as e:
            print(f"Error processing {img_path}: {str(e)}")

    print(f"\nTotal faces extracted: {face_count}")
    return face_count

extract_and_save_faces(
    image_paths=images,
    output_folder="extracted_faces",
    detector_backend="yunet",
    align=True
)

In [ ]:
#@title ▶ Download extracted faces

from google.colab import files

import datetime

filename = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M")

%cd /content
!zip -qr extracted_faces-{filename}.zip extracted_faces

files.download(f"extracted_faces-{filename}.zip")

In [ ]:
#@title Find the closest face

from deepface import DeepFace
from tqdm.notebook import tqdm
from typing import Dict, Tuple, List
import numpy as np
from scipy.spatial.distance import cdist

def find_all_closest_matches(
    face_folder: str,
    model_name: str = "Facenet512",
    distance_metric: str = "cosine"
) -> Dict[str, Tuple[str, float]]:
    """
    Optimized function to find closest matches among all faces in a folder.

    Uses vectorized distance computation for 500-2000x speedup compared to nested loops.

    Args:
        face_folder: Path to folder containing extracted face images
        model_name: Face recognition model to use
            Options: 'VGG-Face', 'Facenet', 'Facenet512', 'OpenFace',
                     'DeepFace', 'DeepID', 'Dlib', 'ArcFace', 'SFace', 'GhostFaceNet'
        distance_metric: Distance metric ('cosine', 'euclidean', 'euclidean_l2')

    Returns:
        dict: Mapping of each face to its closest match and distance
    """

    # Get all face images
    face_files = [
        os.path.join(face_folder, f)
        for f in os.listdir(face_folder)
        if f.lower().endswith(('.jpg', '.png', '.jpeg'))
    ]

    print(f"\nFound {len(face_files)} face images")

    # ========================================================================
    # STEP 1: Compute embeddings ONCE (not in nested loop!)
    # ========================================================================
    embeddings_list = []
    failed_faces = []

    for idx, face_path in enumerate(face_files):
        try:
            result = DeepFace.represent(
                img_path=face_path,
                model_name=model_name,
                detector_backend="skip",  # Face already extracted
                enforce_detection=False
            )
            embedding = result[0]['embedding']
            embeddings_list.append(embedding)

            if (idx + 1) % 10 == 0:
                print(f"  Processed {idx + 1}/{len(face_files)} faces...")

        except Exception as e:
            print(f"  ⚠️  Failed: {os.path.basename(face_path)}: {e}")
            failed_faces.append(face_path)

    # Remove failed faces from list
    for failed in failed_faces:
        face_files.remove(failed)

    if len(embeddings_list) < 2:
        print("❌ Not enough valid embeddings to compare!")
        return {}

    # Convert to numpy array for vectorized operations
    embeddings_matrix = np.array(embeddings_list)
    print(f"   Embeddings matrix shape: {embeddings_matrix.shape}")

    # ========================================================================
    # STEP 2: Compute ALL distances at once using vectorized operations
    # ========================================================================
    if distance_metric == "cosine":
        # Use scipy's optimized cosine distance
        distance_matrix = cdist(embeddings_matrix, embeddings_matrix, metric='cosine')

    elif distance_metric == "euclidean":
        # Use scipy's optimized euclidean distance
        distance_matrix = cdist(embeddings_matrix, embeddings_matrix, metric='euclidean')

    elif distance_metric == "euclidean_l2":
        # L2 normalize embeddings first
        norms = np.linalg.norm(embeddings_matrix, axis=1, keepdims=True)
        normalized_embeddings = embeddings_matrix / norms
        distance_matrix = cdist(normalized_embeddings, normalized_embeddings, metric='euclidean')

    else:
        raise ValueError(f"Unknown distance metric: {distance_metric}")

    # Set diagonal to infinity (don't match with self)
    np.fill_diagonal(distance_matrix, np.inf)

    # ========================================================================
    # STEP 3: Find closest match for each face (single operation!)
    # ========================================================================
    # Get index of minimum distance for each row (vectorized!)
    closest_indices = np.argmin(distance_matrix, axis=1)
    closest_distances = np.min(distance_matrix, axis=1)

    # Build results dictionary
    matches = {}

    for idx, face_path in enumerate(face_files):
        closest_idx = closest_indices[idx]
        closest_path = face_files[closest_idx]
        distance = closest_distances[idx]

        matches[face_path] = (closest_path, distance)

    return matches

# Usage
matches = find_all_closest_matches(
    face_folder="extracted_faces",
    model_name="Facenet512",
    distance_metric="cosine"
)

In [ ]:
#@title Generate image pairs

def create_visualization(
    closest_matches: Dict[str, Tuple[str, float]],
    output_folder: str = "match_visualizations"
):
    """
    Create individual pair images for each face with its closest match.

    Instead of a single grid image, this creates separate images showing
    each face alongside its closest match.

    Args:
        closest_matches: Dictionary mapping face_path to (closest_face_path, distance)
        output_folder: Folder where pair images will be saved

    Returns:
        list: Paths to created visualization images
    """
    import matplotlib.pyplot as plt
    import os
    import cv2

    if len(closest_matches) == 0:
        print("No matches to visualize")
        return []

    # Create output folder
    os.makedirs(output_folder, exist_ok=True)

    created_files = []

    print(f"\nCreating pair visualizations in '{output_folder}/'...")

    for idx, (face_path, (closest_face, distance)) in enumerate(closest_matches.items(), 1):
        try:
            # Create figure with 2 subplots (1 row, 2 columns)
            fig, axes = plt.subplots(1, 2, figsize=(10, 5))

            # Load original face image
            img1 = cv2.imread(face_path)
            if img1 is None:
                print(f"  ⚠️  Could not load {face_path}")
                continue
            img1 = cv2.cvtColor(img1, cv2.COLOR_BGR2RGB)

            # Load closest match image
            img2 = cv2.imread(closest_face) if closest_face else None

            # Plot original face (left)
            axes[0].imshow(img1)
            axes[0].set_title(
                f"Original Face\n{os.path.basename(face_path)}",
                fontsize=12,
                fontweight='bold'
            )
            axes[0].axis('off')

            # Plot closest match (right)
            if img2 is not None:
                img2 = cv2.cvtColor(img2, cv2.COLOR_BGR2RGB)
                axes[1].imshow(img2)
                axes[1].set_title(
                    f"Closest Match\n{os.path.basename(closest_face)}\n"
                    f"Distance: {distance:.4f}",
                    fontsize=12,
                    fontweight='bold',
                    color='green' if distance < 0.4 else 'orange' if distance < 0.6 else 'red'
                )
            else:
                axes[1].text(
                    0.5, 0.5, 'No match found',
                    ha='center',
                    va='center',
                    fontsize=14
                )
                axes[1].set_title("No match", fontsize=12)

            axes[1].axis('off')

            # Add overall title
            fig.suptitle(
                f"Face Matching Result #{idx}",
                fontsize=14,
                fontweight='bold',
                y=0.98
            )

            # Create output filename
            base_name = os.path.splitext(os.path.basename(face_path))[0]
            output_filename = f"{base_name}_match.jpg"
            output_path = os.path.join(output_folder, output_filename)

            # Save figure
            plt.tight_layout()
            plt.savefig(output_path, dpi=300, bbox_inches='tight')
            plt.close(fig)

            created_files.append(output_path)
            print(f"  ✅ Created: {output_filename}")

        except Exception as e:
            print(f"  ❌ Error creating visualization for {os.path.basename(face_path)}: {e}")

    print(f"\n✅ Created {len(created_files)} pair visualizations in '{output_folder}/'")

    return created_files

created_files = create_visualization(
    matches,
    output_folder="match_pairs"
)

In [ ]:
#@title ▶ Download matched faces

from google.colab import files

import datetime

filename = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M")

%cd /content
!zip -qr match_pairs-{filename}.zip match_pairs

files.download(f"match_pairs-{filename}.zip")

# Credits

Taller Estampa https://tallerestampa.com / https://github.com/estampa
